In [8]:
%load_ext sql
%sql duckdb:///rank_scores.duckdb

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [9]:
%%sql
Create table If Not Exists Scores (id int, score DECIMAL(3,2));
Truncate table Scores;
insert into Scores (id, score) values ('1', '3.5');
insert into Scores (id, score) values ('2', '3.65');
insert into Scores (id, score) values ('3', '4.0');
insert into Scores (id, score) values ('4', '3.85');
insert into Scores (id, score) values ('5', '4.0');
insert into Scores (id, score) values ('6', '3.65');

 * duckdb:///rank_scores.duckdb
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.


Count


In [21]:
%%sql

SELECT *
FROM Scores
GROUP BY score
ORDER BY score DESC, id ASC;

 * duckdb:///rank_scores.duckdb
(_duckdb.BinderException) Binder Error: column "id" must appear in the GROUP BY clause or must be part of an aggregate function.
Either add it to the GROUP BY list, or use "ANY_VALUE(id)" if the exact value of "id" is not important.
[SQL: SELECT *
FROM Scores
GROUP BY score
ORDER BY score DESC, id ASC;]
(Background on this error at: https://sqlalche.me/e/20/f405)


In [5]:
%%sql
SELECT 
    s1.score,
    (
        SELECT COUNT(DISTINCT s2.score)
        FROM Scores s2
        WHERE s2.score >= s1.score
    ) as rank
FROM Scores s1
ORDER BY s1.score DESC;

 * duckdb:///rank_scores.duckdb
Done.


score,rank


### Modern Practice Notes (Ranking vs ORDER BY)

- Use ranking (`DENSE_RANK`, `RANK`, `ROW_NUMBER`) when rank is business logic.
- Use final `ORDER BY` when you only need display order.
- Keep reusable logic in a view/CTE/subquery; do presentation sorting in the outer query.
- Always add tie-breakers for deterministic ordering (for example: `ORDER BY score DESC, id ASC`).
- In SQLAlchemy, compute ranking in a CTE/subquery, then filter/sort in the outer query.


In [ ]:
# =========================================================
# SQL Ranking + ORDER BY Guide (examples + when to use)
# =========================================================

# 1) DENSE_RANK(): 
#    - Same value => same rank, no gaps
#    - Example ranks: 1, 1, 2, 3, 3, 4
#    - Use when ties should share rank and rank numbers must be consecutive.
SELECT
  id,
  score,
  DENSE_RANK() OVER (ORDER BY score DESC) AS dense_rank_value
FROM Scores
ORDER BY score DESC;

# 2) RANK(): 
#    - Same value => same rank, with gaps after ties
#    - Example ranks: 1, 1, 3, 4, 4, 6
#    - Use for competition-style ranking where ties skip positions.
SELECT
  id,
  score,
  RANK() OVER (ORDER BY score DESC) AS rank_value
FROM Scores
ORDER BY score DESC;

# 3) ROW_NUMBER(): 
#    - Every row gets a unique sequence number
#    - Example numbers: 1, 2, 3, 4, 5, 6
#    - Use when you need exactly one position per row (no shared ranks).
#    - Add a tie-breaker (like id) for deterministic ordering.
SELECT
  id,
  score,
  ROW_NUMBER() OVER (ORDER BY score DESC, id ASC) AS row_num
FROM Scores
ORDER BY score DESC, id ASC;

# 4) OVER (ORDER BY ...): 
#    - Defines ranking direction inside window functions
#    - DESC: higher score gets better rank.
#    - ASC : lower score gets better rank.
SELECT
  id,
  score,
  DENSE_RANK() OVER (ORDER BY score DESC) AS r
FROM Scores;

# 5) ORDER BY (final query): 
#    - Controls output display order
#    - ORDER BY at the end sorts returned rows.
#    - It does not change rank logic already defined inside OVER(...).
SELECT
  id,
  score,
  DENSE_RANK() OVER (ORDER BY score DESC) AS r
FROM Scores
ORDER BY r ASC, id ASC;

# 6) Fallback without window functions (older SQL engines)
#    - Rank = number of distinct scores >= current score
#    - Use only if window functions are unavailable.
#    - Usually slower and less readable than DENSE_RANK().
SELECT
  s1.score,
  (
    SELECT COUNT(DISTINCT s2.score)
    FROM Scores s2
    WHERE s2.score >= s1.score
  ) AS rank_value
FROM Scores s1
ORDER BY s1.score DESC;

# Quick pick for LeetCode "Rank Scores":
# DENSE_RANK() OVER (ORDER BY score DESC)


SyntaxError: unmatched ')' (1092122437.py, line 6)

# Short version: teams usually separate derived metrics from presentation order.

  1. Use ranking (row_number, rank, dense_rank) when rank itself is business logic.

  - Leaderboards
  - “Top 3 per category”
  - Dedup keep-latest row per key
  - Percentiles/cohorts

  2. Use plain ORDER BY when you only need display order.

  - Sort newest first
  - Sort by score/name/date for UI tables
  - No rank column needed

  3. View practice (PostgreSQL/startups):

  - Put reusable logic in a view/CTE (including rank if needed).
  - Do final ORDER BY in the outer query (app/API layer).
  - Avoid relying on “natural order” from a view.
  - Include tie-breakers (ORDER BY score DESC, id ASC) for deterministic results.

  4. SQLAlchemy practice:

  - Build ranking in SQLAlchemy expression layer, usually subquery/CTE.
  - Then filter/order in outer query.
  - Keep pagination separate (prefer keyset/cursor for large tables).

  Example pattern in SQLAlchemy:

  from sqlalchemy import select, func

  ranked = (
      select(
          Scores.c.id,
          Scores.c.score,
          func.dense_rank().over(order_by=Scores.c.score.desc()).label("rank"),
      ).cte("ranked")
  )

  q = (
      select(ranked.c.id, ranked.c.score, ranked.c.rank)
      .order_by(ranked.c.rank.asc(), ranked.c.id.asc())  # presentation order
  )

  Rule of thumb:

  - If product needs “position/standing,” compute rank.
  - If product only needs row order, just ORDER BY.